# Week 08 — Capstone: reproducible Python-to-AI workflow

**Seismic Python & AI Engineering | Hands-on notebook**

Bring the engineering, testing, ML validation and UI principles together into a small, reviewable demonstration project.

**Estimated independent study:** 4–5 hours in this notebook, plus 2–3 hours on repository exercises and review.  
**Data:** entirely synthetic; no external API, proprietary dataset, or GPU required.  
**Prerequisite:** complete the root README setup and open Jupyter from the repository root.

### Learning outcomes
- Execute the synthetic seismic signal pipeline and explain its assumptions.
- Produce an ML validation summary without train/test group leakage.
- Persist reproducibility metadata and a concise machine-readable report.
- Prepare a pull request containing tested code, evidence and limitations.

### How to study
Read the explanations, execute the worked cells in order, complete the **Your turn** tasks, and implement the week's function in `src/seismic_toolkit/challenges.py`. Run the targeted exercise test to assess progress. The initial baseline tests should pass even before you complete the challenges.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from seismic_toolkit import __version__
print(f"Seismic toolkit version: {__version__}")
np.set_printoptions(precision=4, suppress=True)

## 1. Capstone brief
Imagine a team wants an educational prototype demonstrating (a) baseline/monitor time-shift estimation, (b) synthetic facies classification, and (c) a readable report. Your deliverable is **not** a production seismic interpretation engine; it is a coherent, well-tested software package with defensible data handling.

Acceptance criteria:
1. Script or CLI reproducibly generates the signal example.
2. Grouped ML evaluation reports balanced accuracy and macro F1.
3. All package baseline tests pass.
4. A JSON manifest records seed, dataset size and grouped accuracy statistics.
5. README includes how to run, assumptions and limitations.

In [ ]:
from seismic_toolkit.pipeline import run_synthetic_demo
from seismic_toolkit.datasets import make_synthetic_facies_dataset
from seismic_toolkit.ml import evaluate_group_holdout, grouped_cv_accuracy

signals = run_synthetic_demo(dt=0.001, delay_samples=4)
print("Time shift: expected", signals["true_delay_s"], "estimated", signals["estimated_delay_s"])
frame = make_synthetic_facies_dataset(n_groups=12, samples_per_group=50, seed=42)
metrics = evaluate_group_holdout(frame, seed=42)
scores = grouped_cv_accuracy(frame, n_splits=4)
print("Grouped test metrics:", {k: round(metrics[k], 3) for k in ["accuracy", "balanced_accuracy", "macro_f1"]})
print("Group CV scores:", np.round(scores, 3))

## 2. Check invariants before making a report
A polished plot cannot repair an incorrect experimental design. Confirm physical units, deterministic seed, group isolation and type conversion. Decide what belongs in the output: metrics and configuration can be public; real seismic volumes and private identifiers usually cannot.

In [ ]:
assert np.isclose(signals["true_delay_s"], signals["estimated_delay_s"])
assert not set(metrics["train_groups"]).intersection(metrics["test_groups"])
assert len(scores) == 4 and np.all(np.isfinite(scores))
print("Core numerical and validation invariants: PASS")

## 3. Independent implementation — Week 8 challenge
Implement `build_run_manifest(seed, n_samples, scores)` in `challenges.py`.

Return a JSON-safe dictionary with `seed` (int), `n_samples` (int), `mean_accuracy` (float) and `std_accuracy` (float, **population** standard deviation). Validate nonempty finite scores between zero and one, and positive sample count.

**Your turn A:** add an ISO-style timestamp as an *optional* field without affecting reproducibility comparisons. Explain which metadata must be fixed to reproduce the experiment.

In [ ]:
import json
report = {
    "purpose": "training prototype only",
    "seed": 42,
    "n_samples": int(len(frame)),
    "mean_accuracy": float(np.mean(scores)),
    "std_accuracy": float(np.std(scores)),
    "group_holdout_macro_f1": float(metrics["macro_f1"]),
    "shift_ms": float(signals["estimated_delay_s"] * 1000),
    "limitations": ["Synthetic data", "Not validated on field or reservoir observations"],
}
print(json.dumps(report, indent=2))

## 4. Export a reproducible artifact
Write summary data, not raw seismic samples. Use a clearly named output folder ignored by Git. A reporting module should eventually handle I/O, while a scientific function returns in-memory results.

**Your turn B:** write a report schema test checking required fields, numeric types and absence of unexpected personal or proprietary data.

In [ ]:
from pathlib import Path
root = Path.cwd()
if root.name == "notebooks": root = root.parent
output_dir = root / "outputs"
output_dir.mkdir(exist_ok=True)
report_path = output_dir / "capstone_demo_report.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Wrote:", report_path)
assert json.loads(report_path.read_text(encoding="utf-8"))["seed"] == 42

## 5. Capstone review rubric (100 points)
- **Correctness and numerical tests (25):** sign, units, shapes, invalid inputs.
- **Software engineering (20):** modules, small functions, typing, logging, Git hygiene.
- **ML validation (20):** grouped split, training-only preprocessing, metrics and limitations.
- **AI-assisted development process (15):** written spec, human review, test evidence, confidentiality.
- **Reproducibility and documentation (10):** environment, seeds, assumptions, usage commands.
- **Communication and delivery (10):** concise README, change summary, clear presentation.

**Applied problem:** ask a colleague to use only the README to install and run your prototype. Record every unclear step as an issue, then fix the documentation.

## 6. Engineering handoff checklist
- [ ] `python -m pytest -q` passes.
- [ ] `python -m pytest -o addopts='' -m exercise -q` passes after all exercises are implemented.
- [ ] `python -m seismic_toolkit.cli --delay-samples 4` prints a valid summary.
- [ ] `streamlit run apps/streamlit_app.py` works after installing the optional dependency.
- [ ] No private data, credentials, or huge generated artifacts are committed.
- [ ] The pull request explains limitations and includes a minimal reproducible example.

**Stretch project:** design a new attribute extraction module with a documented numerical contract, independent unit tests, and a simple UI visualization. Do not claim geological validity without representative field validation.

---

## Weekly submission and self-review

1. Complete the independent exercise in `src/seismic_toolkit/challenges.py`.
2. Run: `python -m pytest -o addopts='' -m exercise -k week8 -q`.
3. Run baseline regression tests: `python -m pytest -q`.
4. Commit a focused change to a feature branch; document what changed and why.
5. Record one failure mode, one design decision, and one next-step question in your learning journal.

**Further work:** see [`../exercises/week08.md`](../exercises/week08.md) for a graded problem, extra practice and acceptance criteria. Compare with `solutions/reference_challenges.py` only **after** attempting your own solution.

**Important:** all datasets and algorithms are educational demonstrations, not validated reservoir-interpretation systems.